# Pretraining a GPT with matrixgrad

Trains a 16M-parameter GPT on GPT-2 BPE tokens using only `matrixgrad` for autograd. Runs on CuPy if it's installed, NumPy otherwise.

- 4 layers, 4 heads, 256-dim embeddings, context of 256, tied input/output embeddings
- AdamW, with weight decay on matrices only
- linear warmup then cosine decay
- gradient clipping at 1.0
- GPT-2 init, with residual projections scaled by `1 / sqrt(2 * n_layer)`

Run `python data/prepare_openwebtext.py` or `python data/prepare_wikitext.py` first and set `DATASET` to match.

## Setup

In [ ]:
import os, time, pickle, math
import numpy as onp
import tiktoken

from matrixgrad.backend import np as xp, GPU
from matrixgrad.engine import no_grad
from matrixgrad.nn import GPT, gpt2_init_
from matrixgrad.optimizer import AdamW, clip_grad_norm_

onp.random.seed(1337)
if GPU:
    xp.random.seed(1337)

# CuPy's memory pool fragments over a long run, so release it every so often
if GPU:
    import cupy as _cp
    def free_pool():
        _cp.get_default_memory_pool().free_all_blocks()
        _cp.get_default_pinned_memory_pool().free_all_blocks()
else:
    def free_pool():
        pass

DATASET  = 'openwebtext'  # 'openwebtext' or 'wikitext'
DATA_DIR = os.path.join(os.getcwd(), 'data', DATASET)
CKPT_DIR = os.path.join(os.getcwd(), 'checkpoints')
os.makedirs(CKPT_DIR, exist_ok=True)

enc = tiktoken.get_encoding('gpt2')
vocab_size = enc.n_vocab

print(f'backend:    {xp.__name__}  (GPU={GPU})')
print(f'dataset:    {DATASET}')
print(f'vocab_size: {vocab_size}')

## Data

`train.bin` and `val.bin` are flat uint16 arrays of token ids. They're memory-mapped, so only the sampled windows are read into RAM.

In [ ]:
train_path = os.path.join(DATA_DIR, 'train.bin')
val_path   = os.path.join(DATA_DIR, 'val.bin')
assert os.path.exists(train_path), f'missing {train_path}: run data/prepare_{DATASET}.py first'

train_data = onp.memmap(train_path, dtype=onp.uint16, mode='r')
val_data   = onp.memmap(val_path,   dtype=onp.uint16, mode='r')
print(f'train: {len(train_data):,} tokens   val: {len(val_data):,} tokens')
print('first 30 train tokens decoded:')
print(enc.decode(train_data[:30].tolist()))

In [4]:
def get_batch(split: str, batch_size: int, block_size: int):
    src = train_data if split == 'train' else val_data
    ix = onp.random.randint(0, len(src) - block_size - 1, size=(batch_size,))
    x = onp.stack([src[i:i+block_size].astype(onp.int64) for i in ix])
    y = onp.stack([src[i+1:i+block_size+1].astype(onp.int64) for i in ix])
    if GPU:
        x, y = xp.asarray(x), xp.asarray(y)
    return x, y

xb, yb = get_batch('train', batch_size=4, block_size=16)
print('xb:', xb.shape, xb.dtype)
print('yb:', yb.shape, yb.dtype)
print('xb[0]:', xb[0])
print('decoded:', repr(enc.decode(onp.asarray(xb[0] if not GPU else xb[0].get()).tolist())))

xb: (4, 16) int64
yb: (4, 16) int64
xb[0]: [ 2383 49936    13  2329  1410   319  2491  2793  3081  6460   319   749
  3341   290   345  1183]
decoded: " significant bottleneck. Just plan on running lower quality settings on most systems and you'll"


## Hyperparameters

Model size barely affects throughput here. The per-op Python overhead of the autograd engine dominates, not the GPU.

In [ ]:
# model
n_layer    = 4
n_head     = 4
n_embd     = 256
block_size = 256
dropout    = 0.0

# optim
# the (batch * block, vocab) logits tensor is ~200 MB at batch_size=4; drop to 2 if you OOM
batch_size      = 4
max_iters       = 20000
warmup_iters    = 200
max_lr          = 3e-4
min_lr          = 3e-5
weight_decay    = 0.1
betas           = (0.9, 0.95)
grad_clip       = 1.0

# eval / logging / ckpt
eval_interval   = 500
eval_iters      = 20
log_interval    = 100

## Model

In [ ]:
model = GPT(vocab_size, n_embd, n_head, n_layer, block_size, dropout, tie_weights=True)
gpt2_init_(model)

n_params = sum(int(onp.prod(p.shape)) for p in model.parameters())
n_untied = (n_params + vocab_size * n_embd) / 1e6
print(f'parameters: {n_params/1e6:.2f}M  (tied; without tying would be ~{n_untied:.2f}M)')

# initial loss should be close to log(vocab_size)
xb, yb = get_batch('val', batch_size=2, block_size=32)
_, loss0 = model(xb, yb)
print(f'init loss:  {float(loss0.data):.4f}  (expected ~log({vocab_size}) = {onp.log(vocab_size):.4f})')

## Optimizer

Matrices and embeddings get weight decay; biases and LayerNorm parameters don't. Two `AdamW` instances over disjoint parameter lists handle the split.

In [7]:
params = model.parameters()
decay_params    = [p for p in params if p.data.ndim >= 2]
no_decay_params = [p for p in params if p.data.ndim <  2]
print(f'decay group:    {len(decay_params):3d} tensors  ({sum(int(onp.prod(p.shape)) for p in decay_params)/1e6:.2f}M params)')
print(f'no-decay group: {len(no_decay_params):3d} tensors  ({sum(int(onp.prod(p.shape)) for p in no_decay_params)/1e6:.2f}M params)')

opt_decay    = AdamW(decay_params,    lr=max_lr, betas=betas, weight_decay=weight_decay)
opt_no_decay = AdamW(no_decay_params, lr=max_lr, betas=betas, weight_decay=0.0)

def get_lr(step: int) -> float:
    if step < warmup_iters:
        return max_lr * (step + 1) / (warmup_iters + 1)
    if step >= max_iters:
        return min_lr
    decay_ratio = (step - warmup_iters) / (max_iters - warmup_iters)
    coeff = 0.5 * (1 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (max_lr - min_lr)

def set_lr(lr: float):
    opt_decay.lr = lr
    opt_no_decay.lr = lr

decay group:     18 tensors  (16.08M params)
no-decay group:  35 tensors  (0.06M params)


## Eval

In [8]:
def estimate_loss():
    model.eval()
    out = {}
    with no_grad():
        for split in ('train', 'val'):
            losses = onp.empty(eval_iters)
            for k in range(eval_iters):
                xb, yb = get_batch(split, batch_size, block_size)
                _, loss = model(xb, yb)
                losses[k] = float(loss.data)
            out[split] = losses.mean()
    model.train()
    return out

## Checkpointing

Parameters and optimizer state are copied to host memory before pickling, so checkpoints load on either backend.

In [9]:
def _to_host(a):
    return a.get() if hasattr(a, 'get') else a

def _to_device(a):
    return xp.asarray(a) if GPU else a

def save_ckpt(path, step, val_loss):
    state = {
        'step': step,
        'val_loss': val_loss,
        'config': dict(n_layer=n_layer, n_head=n_head, n_embd=n_embd,
                       block_size=block_size, vocab_size=vocab_size, dropout=dropout),
        'params': [_to_host(p.data) for p in model.parameters()],
        'opt_decay': dict(
            t=opt_decay.t,
            m=[_to_host(m) for m in opt_decay.m],
            v=[_to_host(v) for v in opt_decay.v],
        ),
        'opt_no_decay': dict(
            t=opt_no_decay.t,
            m=[_to_host(m) for m in opt_no_decay.m],
            v=[_to_host(v) for v in opt_no_decay.v],
        ),
    }
    with open(path, 'wb') as f:
        pickle.dump(state, f)
    print(f'  ckpt -> {path}  ({os.path.getsize(path)/1e6:.1f} MB)')

def load_ckpt(path):
    with open(path, 'rb') as f:
        state = pickle.load(f)
    for p, arr in zip(model.parameters(), state['params']):
        p.data = _to_device(arr)
    opt_decay.t = state['opt_decay']['t']
    opt_decay.m = [_to_device(a) for a in state['opt_decay']['m']]
    opt_decay.v = [_to_device(a) for a in state['opt_decay']['v']]
    opt_no_decay.t = state['opt_no_decay']['t']
    opt_no_decay.m = [_to_device(a) for a in state['opt_no_decay']['m']]
    opt_no_decay.v = [_to_device(a) for a in state['opt_no_decay']['v']]
    return state['step'], state['val_loss']

## Train

In [ ]:
best_val = float('inf')
t0 = time.perf_counter()

for step in range(max_iters):
    set_lr(get_lr(step))

    # eval + checkpoint
    if step % eval_interval == 0 or step == max_iters - 1:
        free_pool()
        losses = estimate_loss()
        elapsed = time.perf_counter() - t0
        print(f'step {step:5d}  train {losses["train"]:.4f}  val {losses["val"]:.4f}  lr {opt_decay.lr:.2e}  ({elapsed:.1f}s)')
        if losses['val'] < best_val:
            best_val = losses['val']
            save_ckpt(os.path.join(CKPT_DIR, 'best.pkl'), step, best_val)
        free_pool()
    elif step % log_interval == 0:
        elapsed = time.perf_counter() - t0
        print(f'step {step:5d}  loss {float(loss.data):.4f}  lr {opt_decay.lr:.2e}  ({elapsed:.1f}s)')

    # train step
    xb, yb = get_batch('train', batch_size, block_size)
    _, loss = model(xb, yb)
    opt_decay.zero_grad()
    opt_no_decay.zero_grad()
    loss.backward()
    clip_grad_norm_(params, grad_clip)
    opt_decay.step()
    opt_no_decay.step()

    if GPU and step % 50 == 0:
        free_pool()

print(f'\ndone, {time.perf_counter()-t0:.1f}s total, best val {best_val:.4f}')

## Sample

In [11]:
model.eval()
prompt = 'The history of'
prompt_ids = onp.array([enc.encode(prompt)], dtype=onp.int64)
if GPU:
    prompt_ids = xp.asarray(prompt_ids)

out = model.generate(prompt_ids, max_new_tokens=200, temperature=0.8, top_k=50)
out_host = out.get() if hasattr(out, 'get') else out
print(enc.decode(out_host[0].tolist()))

The history of the most interesting world, and its current world-game world. The most important thing is the world is to come in the world, and it’s a great job that is to take the first time for the American world. It’s the most popular man who has the world and will be the same for it. However, these are the new world, which is the whole world of modern, and its history of the world. It’s a way to the city it’s history. The world in the world is that they are on many of the world, and on the world. “I will still want to do that now,” says the world is a great one is a great world.

The book is a world very different, from the world, and it is my very small world to be a great man, too. It’s that bad, it’s the world. The problem is a way to make


In [12]:
model.eval()
prompt = 'Computer science would not be what it is today without'
prompt_ids = onp.array([enc.encode(prompt)], dtype=onp.int64)
if GPU:
    prompt_ids = xp.asarray(prompt_ids)

out = model.generate(prompt_ids, max_new_tokens=200, temperature=0.8, top_k=50)
out_host = out.get() if hasattr(out, 'get') else out
print(enc.decode(out_host[0].tolist()))

Computer science would not be what it is today without what we do?

A recent article: “That is not a good deal, if you’re going to take on the rest of the business that you’re going to be going,” he said.

The issue was to be if you’re at this point, it’s something to do is going to be the way to see the information.

“This is what the American people want to do with us,” he said. “I’m not trying to make it a chance.

‘I’m an extremely common day?’” he said, adding that “is the problem in this, it’s a man who gets out of the right person,” he said.

A full-term report said there’s no way to make it an event. A few of our own things are going to move forward to a little
